In [8]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER
import os

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'Inbursa', 'Inbursa_credit_statement.pdf')
statement_path

'c:\\Proyectos\\Aether\\Aether_v1\\documents\\inputs\\test_files\\Inbursa\\Inbursa_credit_statement.pdf'

In [9]:
from document_reader import PDFReader

reader = PDFReader(statement_path)
document_height = reader.get_height()
extracted_words = reader.extract_words_from_pdf()

extracted_words

,page,text,x0,top,x1,bottom
0,1,Página,516.167000,15.432429,541.696875,25.432380
1,1,1,543.976864,15.432429,548.536841,25.432380
2,1,de,550.816830,15.432429,559.936786,25.432380
3,1,4,562.216774,15.432429,566.776752,25.432380
4,1,ESTADO,407.210890,29.788711,461.882852,45.788700
...,...,...,...,...,...,...
1267,4,GENERAL,283.402316,777.045667,307.848022,782.045812
1268,4,DE,309.238062,777.045667,316.183263,782.045812
1269,4,LEY,317.573303,777.045667,327.303585,782.045812
1270,4,PERSONAS,328.693625,777.045667,356.754436,782.045812


In [10]:
from bank_detector import DefaultBankDetector

bank_detector = DefaultBankDetector(extracted_words, document_height)

bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.statement_propertys

print(f'{bank} - {statement_type}')

inbursa - credit


In [11]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table

1269
261 - 473


,page,text,x0,top,x1,bottom
0,1,Cantidad,498.893170,471.126977,538.912974,481.126928
1,1,Fecha,39.544560,478.213753,67.334424,488.213704
2,1,Descripción,250.259900,478.213753,302.489644,488.213704
3,1,(pesos),502.233133,482.627276,535.572969,492.627227
4,1,10/23,41.555163,503.132700,65.323750,512.632536
...,...,...,...,...,...,...
207,2,cargos:,94.589878,34.107225,125.739837,43.607060
208,2,"$4,674.92",128.380792,34.107225,170.636059,43.607060
209,2,Total,241.941822,34.107225,263.059955,43.607060
210,2,abonos:,265.700909,34.107225,299.501323,43.607060


In [12]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table, statement_properties)

header_row = row_segmenter.get_header_row()
grouped_rows = row_segmenter.group_rows()

print(row_segmenter.row_threshold)
print(header_row)
grouped_rows

7.912954233333338
{'columns': ['Fecha', 'Descripción', 'Cantidad (pesos)'], 'x0': [39.54456000000002, 250.25990000000002, 498.89317000000005], 'x1': [67.33442382900002, 302.489644073, 535.572969334]}


,row_group,text,words,top,bottom,page
0,0,Cantidad Fecha Descripción (pesos),"[(Cantidad, 498.89317000000005, 538.912973902)...",471.126977,492.627227,1
1,1,10/23 MERCADO PAGO - MER 991006JMA - Consumo e...,"[(10/23, 41.55516270000004, 65.32375037040003)...",503.132700,512.632536,1
2,2,11/03 Cargo por Anualidad - (3/3) $0.06,"[(11/03, 41.55515970000005, 65.32374737040004)...",513.753741,523.253577,1
3,3,11/03 I.V.A. del 16% de Cargo por Anualidad - ...,"[(11/03, 41.5551567, 65.3237443704), (I.V.A., ...",528.004001,537.503837,1
4,4,12/16 MERCADO PAGO - MER 991006JMA - Consumo c...,"[(12/16, 41.55516369999998, 65.32375137039998)...",538.624704,548.124540,1
5,5,12/25 SUPERCENTER NICOLAS RO - NWM 9709244W4 -...,"[(12/25, 41.555160700000044, 65.32374837040004...",552.874964,562.374800,1
6,6,12/28 MERCADO PAGO - MER 991006JMA - Consumo c...,"[(12/28, 41.55515770000011, 65.3237453704001),...",563.496005,572.995841,1
7,7,12/30 GEM E COM - GEM 850101BJ3 - Consumo cont...,"[(12/30, 41.55515470000017, 65.32374237040017)...",577.745927,587.245763,1
8,8,01/03 Su Pago en Wal Mart Gracias - En Efectiv...,"[(01/03, 41.55515170000024, 65.32373937040023)...",588.366968,597.866804,1
9,9,01/05 CDMX E COM - GDF 9712054NA - Consumo con...,"[(01/05, 41.55515870000022, 65.32374637040022)...",602.616890,612.116726,1


In [13]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, header_row, statement_properties)
df_structured = table_reconstructor.get_structured_table()
df_structured

,Fecha,Descripción,Cantidad (pesos)
0,10/23,MERCADO PAGO - MER 991006JMA - Consumo en 3 Cu...,$433.00
1,11/03,Cargo por Anualidad - (3/3),$0.06
2,11/03,I.V.A. del 16% de Cargo por Anualidad - (3/3),$0.00
3,12/16,MERCADO PAGO - MER 991006JMA - Consumo contado,$389.00
4,12/25,SUPERCENTER NICOLAS RO - NWM 9709244W4 - Consu...,"$2,566.86"
5,12/28,MERCADO PAGO - MER 991006JMA - Consumo contado,$399.00
6,12/30,GEM E COM - GEM 850101BJ3 - Consumo contado,$48.00
7,01/03,Su Pago en Wal Mart Gracias - En Efectivo,"$-1,285.00"
8,01/05,CDMX E COM - GDF 9712054NA - Consumo contado,$74.54
9,01/05,CDMX E COM - GDF 9712054NA - Consumo contado,$74.54


In [14]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_structured, extracted_words, statement_properties)

normalizer.normalize_table()

,Date,Description,Amount,Type
0,2019-10-23,MERCADO PAGO - MER 991006JMA - Consumo en 3 Cu...,-433.00,Cargo
1,2019-11-03,Cargo por Anualidad - (3/3),-0.06,Cargo
2,2019-11-03,I.V.A. del 16% de Cargo por Anualidad - (3/3),-0.00,Cargo
3,2019-12-16,MERCADO PAGO - MER 991006JMA - Consumo contado,-389.00,Cargo
4,2019-12-25,SUPERCENTER NICOLAS RO - NWM 9709244W4 - Consu...,-2566.86,Cargo
5,2019-12-28,MERCADO PAGO - MER 991006JMA - Consumo contado,-399.00,Cargo
6,2019-12-30,GEM E COM - GEM 850101BJ3 - Consumo contado,-48.00,Cargo
7,2019-01-03,Su Pago en Wal Mart Gracias - En Efectivo,1285.00,Abono
8,2019-01-05,CDMX E COM - GDF 9712054NA - Consumo contado,-74.54,Cargo
9,2019-01-05,CDMX E COM - GDF 9712054NA - Consumo contado,-74.54,Cargo
